In [1]:
# IMPORT LIBRERIE
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# =========================
# PARTE 1 - ANALISI PER SINGOLO APPELLO
# =========================

# Inserisci qui il ciclo sugli appelli
# Esempio struttura:

# for appello in lista_appelli:
#     df_appello = df[df['Appello'] == appello]
#     
#     # Statistiche
#     print(df_appello['Voto'].describe())
#     
#     # Grafico distribuzione
#     sns.histplot(df_appello['Voto'], kde=True)
#     plt.title(f'Distribuzione voti - {appello}')
#     plt.show()


In [3]:
# =========================
# PARTE 2 - DATASET COMPLESSIVO
# =========================

# Creazione dataframe finale
df_finale = pd.DataFrame()

# Assicurati che ogni dataframe abbia queste colonne:
# 'Voto', 'Appello'

# Esempio:
# df_temp = ...
# df_temp['Appello'] = 'Gennaio'
# df_finale = pd.concat([df_finale, df_temp], ignore_index=True)

# =========================
# ANALISI COMPLESSIVA
# =========================

# Boxplot voti per appello
sns.boxplot(data=df_finale, x='Appello', y='Voto')
plt.title('Distribuzione voti per appello')
plt.show()


ValueError: Could not interpret value `Appello` for `x`. An entry with this name does not appear in `data`.

In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
import plotly.graph_objects as go

# =========================
# 1) DATI DI ESEMPIO
# =========================

# Simuliamo i tuoi appelli reali
dati = {
    "appello_id": [1, 2, 3],
    "data_appello": ["23/09/2025", "26/01/2026", "17/06/2026"],
    "voto": [24, 22, 20]  # puoi cambiare questi valori per fare prove
}

df = pd.DataFrame(dati)
df["data_appello"] = pd.to_datetime(df["data_appello"], dayfirst=True)

df

,appello_id,data_appello,voto
0,1,2025-09-23,24
1,2,2026-01-26,22
2,3,2026-06-17,20


In [4]:
# =========================
# 2) FUNZIONE DI PREVISIONE
# =========================

def grafico_previsione(df):
    if df.empty or "voto" not in df.columns or "data_appello" not in df.columns:
        print("DF non valido")
        return None

    df = df.copy()

    # Conversioni
    df["voto_num"] = pd.to_numeric(df["voto"], errors="coerce")
    df["data_appello"] = pd.to_datetime(df["data_appello"], errors="coerce")
    df_validi = df.dropna(subset=["voto_num", "data_appello"])

    # Media per appello (qui ogni appello è unico, ma teniamo la logica generale)
    df_media = df_validi.groupby(["appello_id", "data_appello"])["voto_num"].mean().reset_index()
    df_media = df_media.sort_values("data_appello")

    n_appelli = len(df_media)
    if n_appelli < 2:
        print("Dati insufficienti per una previsione")
        return None

    # Asse X = giorni dal primo appello
    df_media["giorni"] = (df_media["data_appello"] - df_media["data_appello"].min()).dt.days
    X = df_media["giorni"].values.reshape(-1, 1)
    y = df_media["voto_num"].values

    # Distanza media tra appelli (minimo 30 giorni)
    distanze = df_media["data_appello"].diff().dt.days.dropna()
    media_distanza = max(30, int(distanze.mean()))

    # Numero previsioni future = numero appelli storici
    n_future = n_appelli

    future_pred = []
    future_dates = []

    current_X = X.copy()
    current_y = y.copy()
    last_date = df_media["data_appello"].max()

    for i in range(n_future):
        model = LinearRegression()
        model.fit(current_X, current_y)

        next_date = last_date + pd.Timedelta(days=media_distanza)
        last_date = next_date
        future_dates.append(next_date)

        next_day = (next_date - df_media["data_appello"].min()).days
        next_pred = model.predict([[next_day]])[0]

        next_pred = float(np.clip(next_pred, 18, 31))

        future_pred.append(next_pred)

        current_X = np.append(current_X, [[next_day]], axis=0)
        current_y = np.append(current_y, next_pred)

    storico_x = df_media["data_appello"]
    storico_y = df_media["voto_num"]

    previsioni_x = pd.to_datetime(future_dates)
    previsioni_y = future_pred

    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=storico_x,
        y=storico_y,
        mode="lines+markers",
        name="Storico",
        line=dict(color="orange")
    ))

    fig.add_trace(go.Scatter(
        x=previsioni_x,
        y=previsioni_y,
        mode="lines+markers",
        name="Previsioni",
        line=dict(color="red")
    ))

    fig.update_layout(
        title="Previsione media voti",
        xaxis_title="Data appello",
        yaxis_title="Media voto",
        yaxis=dict(range=[0, 31])
    )

    fig.show()

    return df_media, list(zip(previsioni_x, previsioni_y))
print(grafico_previsione(df))

(   appello_id data_appello  voto_num  giorni
0           1   2025-09-23      24.0       0
1           2   2026-01-26      22.0     125
2           3   2026-06-17      20.0     267, [(Timestamp('2026-10-28 00:00:00'), 18.0), (Timestamp('2027-03-10 00:00:00'), 18.0), (Timestamp('2027-07-21 00:00:00'), 18.0)])


In [11]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
import plotly.graph_objects as go

def previsione_iterativa_senza_date(voti_reali, n_future):
    """
    voti_reali: lista di voti reali, es. [24, 22, 20]
    n_future: quanti punti prevedere
    """

    voti_reali = list(voti_reali)
    storico_x = list(range(1, len(voti_reali) + 1))
    storico_y = voti_reali.copy()

    # dataset corrente
    current_x = np.array(storico_x).reshape(-1, 1)
    current_y = np.array(storico_y, dtype=float)

    future_x = []
    future_y = []

    for i in range(n_future):
        # regressione lineare aggiornata
        model = LinearRegression()
        model.fit(current_x, current_y)

        # prossimo punto X
        next_x = current_x.max() + 1

        # predizione
        next_y = model.predict([[next_x]])[0]

        # limiti realistici
        next_y = float(np.clip(next_y, 18, 31))

        # salva
        future_x.append(next_x)
        future_y.append(next_y)

        # aggiorna dataset
        current_x = np.append(current_x, [[next_x]], axis=0)
        current_y = np.append(current_y, next_y)

    # grafico
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=storico_x,
        y=storico_y,
        mode="lines+markers",
        name="Storico",
        line=dict(color="orange")
    ))

    fig.add_trace(go.Scatter(
        x=future_x,
        y=future_y,
        mode="lines+markers",
        name="Previsioni iterative",
        line=dict(color="red")
    ))

    fig.update_layout(
        title="Previsione iterativa dei voti (senza date)",
        xaxis_title="Numero appello",
        yaxis_title="Media voto",
        yaxis=dict(range=[0, 31])
    )

    fig.show()

    return future_x, future_y
previsione_iterativa_senza_date([24, 22, 20], n_future=5)
print(previsione_iterativa_senza_date([24, 28, 20], n_future=5))


([np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)], [20.0, 18.0, 18.0, 18.0, 18.0])


In [15]:
import numpy as np
from sklearn.linear_model import LinearRegression
import plotly.graph_objects as go

def previsione_iterativa_senza_date(voti_reali, n_future):
    voti_reali = list(voti_reali)

    storico_x = list(range(1, len(voti_reali) + 1))
    storico_y = voti_reali.copy()

    current_x = storico_x.copy()
    current_y = storico_y.copy()

    future_x = []
    future_y = []

    for i in range(n_future):
        # prendi SOLO gli ultimi 3 punti
        x_window = np.array(current_x[-3:]).reshape(-1, 1)
        y_window = np.array(current_y[-3:], dtype=float)

        model = LinearRegression()
        model.fit(x_window, y_window)

        next_x = current_x[-1] + 1
        next_y = model.predict([[next_x]])[0]

        # limiti realistici
        next_y = float(np.clip(next_y, 18, 31))

        # salva
        future_x.append(next_x)
        future_y.append(next_y)

        # aggiorna dataset
        current_x.append(next_x)
        current_y.append(next_y)

    # grafico
    fig = go.Figure()

    fig.add_trace(go.Scatter(
        x=storico_x,
        y=storico_y,
        mode="lines+markers",
        name="Storico",
        line=dict(color="orange")
    ))

    fig.add_trace(go.Scatter(
        x=future_x,
        y=future_y,
        mode="lines+markers",
        name="Previsioni iterative",
        line=dict(color="red")
    ))

    fig.update_layout(
        title="Previsione iterativa (finestra mobile 3 punti)",
        xaxis_title="Numero appello",
        yaxis_title="Voto",
        yaxis=dict(range=[0, 31])
    )

    fig.show()

    return future_x, future_y

print(previsione_iterativa_senza_date([24, 28, 20], n_future=5))

([4, 5, 6, 7, 8], [20.0, 18.0, 18.0, 18.0, 18.0])
